In [16]:
import pandas as pd

dataFrame = pd.read_csv('car_fuel_efficiency_2026.csv', delimiter=',')
dataFrame.head()
# substitute NA / 0 for non existent values
numeric_cols = dataFrame.select_dtypes(include="number").columns
string_cols = dataFrame.select_dtypes(include=["object", "string"]).columns
display(numeric_cols)
display("null numeric columns :" + str(dataFrame[numeric_cols].isnull().sum()))
display(string_cols)
display("null string columns :" + str(dataFrame[string_cols].isnull().sum()))
dataFrame[numeric_cols] = dataFrame[numeric_cols].fillna(0)
dataFrame[string_cols] = dataFrame[string_cols].fillna("NA")
display("after fixed null numeric columns :" + str(dataFrame[numeric_cols].isnull().sum()))
display("after fixed null string columns :" + str(dataFrame[string_cols].isnull().sum()))

Index(['model_year', 'num_doors', 'engine_displacement', 'num_cylinders',
       'horsepower', 'vehicle_weight', 'acceleration', 'fuel_efficiency_mpg'],
      dtype='str')

'null numeric columns :model_year               0\nnum_doors                0\nengine_displacement      0\nnum_cylinders            0\nhorsepower             877\nvehicle_weight           0\nacceleration           264\nfuel_efficiency_mpg      0\ndtype: int64'

Index(['origin', 'fuel_type', 'drivetrain'], dtype='str')

'null string columns :origin        0\nfuel_type     0\ndrivetrain    0\ndtype: int64'

'after fixed null numeric columns :model_year             0\nnum_doors              0\nengine_displacement    0\nnum_cylinders          0\nhorsepower             0\nvehicle_weight         0\nacceleration           0\nfuel_efficiency_mpg    0\ndtype: int64'

'after fixed null string columns :origin        0\nfuel_type     0\ndrivetrain    0\ndtype: int64'

In [25]:
from sklearn.model_selection import train_test_split
df_full_train, df_test = train_test_split(dataFrame, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

#one hot encoding
from sklearn.feature_extraction import DictVectorizer
dv = DictVectorizer(sparse=False)
x_full_train = dv.fit_transform(df_full_train.drop(columns=['fuel_efficiency_mpg']).to_dict(orient='records'))
y_full_train = df_full_train['fuel_efficiency_mpg']
x_train = dv.fit_transform(df_train.drop(columns=['fuel_efficiency_mpg']).to_dict(orient='records'))
y_train = df_train['fuel_efficiency_mpg']
x_val = dv.transform(df_val.drop(columns=['fuel_efficiency_mpg']).to_dict(orient='records'))
y_val = df_val['fuel_efficiency_mpg']
x_test = dv.transform(df_test.drop(columns=['fuel_efficiency_mpg']).to_dict(orient='records'))
y_test = df_test['fuel_efficiency_mpg']

In [26]:
#Let's train a decision tree regressor to predict the fuel_efficiency_mpg variable.
#Train a model with max_depth=1.
from sklearn.tree import DecisionTreeRegressor

model = DecisionTreeRegressor(max_depth=1)
model.fit(x_train, y_train)
y_val_pred = model.predict(x_val)

#1.Which original feature is used for splitting the data? If you use DictVectorizer,
#  map a one-hot feature such as origin=USA back to its original column name.
split_feature_index = model.tree_.feature[0]
print(split_feature_index)
split_feature_name = dv.feature_names_[split_feature_index]
print(split_feature_name)

9
model_year


In [31]:
'''2.Train a random forest regressor with these parameters:

n_estimators=10
random_state=1
n_jobs=-1 (optional - to make training faster)
What's the RMSE of this model on the validation data?'
'"'''

from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
model.fit(x_train, y_train)
y_val_pred = model.predict(x_val)

from sklearn.metrics import root_mean_squared_error
rmse = root_mean_squared_error(y_val, y_val_pred)
print(rmse)

1.8336350509302553


In [33]:
'''Now let's experiment with the n_estimators parameter

Try these values of this parameter: [10, 50, 100, 150].
Set random_state to 1.
Evaluate the model on the validation dataset.'''

from sklearn.ensemble import RandomForestRegressor

for n_estimators in [10, 50, 100, 150]:
    model = RandomForestRegressor(n_estimators=n_estimators, random_state=1, n_jobs=-1)
    model.fit(x_train, y_train)
    y_val_pred = model.predict(x_val)
    from sklearn.metrics import root_mean_squared_error
    rmse = root_mean_squared_error(y_val, y_val_pred)
    print(f"n_estimators={n_estimators}, RMSE={rmse:.3f}")

n_estimators=10, RMSE=1.834
n_estimators=50, RMSE=1.770
n_estimators=100, RMSE=1.768
n_estimators=150, RMSE=1.768


In [34]:
'''Let's select the best max_depth:

Try different values of max_depth: [10, 15, 20, 25]
For each of these values,
try different values of n_estimators from [10, 50, 100, 150]
calculate the mean RMSE
Fix the random seed: random_state=1'''

from sklearn.ensemble import RandomForestRegressor
for max_depth in [10, 15, 20, 25]:
    rmses = []
    for n_estimators in [10, 50, 100, 150]:
        model = RandomForestRegressor(max_depth=max_depth, n_estimators=n_estimators, random_state=1, n_jobs=-1)
        model.fit(x_train, y_train)
        y_val_pred = model.predict(x_val)
        from sklearn.metrics import root_mean_squared_error
        rmse = root_mean_squared_error(y_val, y_val_pred)
        rmses.append(rmse)
    mean_rmse = sum(rmses) / len(rmses)
    print(f"max_depth={max_depth}, mean RMSE={mean_rmse:.3f}")

max_depth=10, mean RMSE=1.757
max_depth=15, mean RMSE=1.785
max_depth=20, mean RMSE=1.783
max_depth=25, mean RMSE=1.787


In [36]:
'''At each step of the decision tree learning algorithm, it finds the best split. When doing it, we can calculate "gain" - the reduction in impurity before and after the split. This gain is quite useful in understanding what are the important features for tree-based models.

In Scikit-Learn, tree-based models contain this information in the feature_importances_ field.

For this homework question, we'll find the most important feature:

Train the model with these parameters:
n_estimators=10,
max_depth=20,
random_state=1,
n_jobs=-1 (optional)
Get the feature importance information from this model'''
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
model.fit(x_train, y_train)
importances = pd.Series(
    model.feature_importances_,
    index=dv.get_feature_names_out()
)
print(importances.sort_values(ascending=False))

model_year                      0.295822
vehicle_weight                  0.203649
fuel_type=Gasoline              0.160713
horsepower                      0.078254
engine_displacement             0.061008
acceleration                    0.055466
origin=USA                      0.042232
fuel_type=Diesel                0.021128
drivetrain=Front-wheel drive    0.020005
num_doors                       0.019724
fuel_type=Hybrid                0.011685
drivetrain=All-wheel drive      0.007873
num_cylinders                   0.006939
origin=Asia                     0.006479
drivetrain=Rear-wheel drive     0.005636
origin=Europe                   0.003389
dtype: float64


In [40]:
'''Create DMatrix for train and validation
Create a watchlist
Train a model with these parameters for 100 rounds:
xgb_params = {
    'eta': 0.3, 
    'max_depth': 6,
    'min_child_weight': 1,
    
    'objective': 'reg:squarederror',
    'nthread': 8,
    
    'seed': 1,
    'verbosity': 1,
}
Now change eta from 0.3 to 0.1.

'''
import xgboost as xgb
features = dv.get_feature_names_out().tolist()
for eta in [0.3, 0.1]:
    dtrain = xgb.DMatrix(x_train, label=y_train, feature_names=features)
    dval = xgb.DMatrix(x_val, label=y_val, feature_names=features)
    watchlist = [(dtrain, 'train'), (dval, 'val')]
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,
        'objective': 'reg:squarederror',
        'nthread': 8,
        'seed': 1,
        'verbosity': 1,
    }
    evals_result = {}
    model = xgb.train(
        xgb_params, dtrain, num_boost_round=100, evals=watchlist,
        evals_result=evals_result, verbose_eval=False
    )
    print(f"eta={eta}, validation RMSE={evals_result['val']['rmse'][-1]:.3f}")


eta=0.3, validation RMSE=1.826
eta=0.1, validation RMSE=1.725
